<div style="border-bottom:3px solid #c8962d;padding-bottom:12px"><strong>Universidad Externado de Colombia</strong><br>Programa de Ciencia de Datos · Deep Learning<br>Docente: Wilmer Pineda-Ríos</div>

# Semana 1 — Tensores y primer MLP

**Resultado.** Seguir los shapes y parámetros de una red densa, reproducir su forward pass con NumPy y entrenar el modelo equivalente en Keras.

## 1. Preparación reproducible

La práctica debe funcionar en CPU. Fijamos semillas y registramos versiones antes de observar resultados.

In [ ]:
import platform
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from tensorflow import keras

RANDOM_STATE = 42
keras.utils.set_random_seed(RANDOM_STATE)
print({'python': platform.python_version(), 'tensorflow': tf.__version__, 'sklearn': sklearn.__version__})

## 2. Libro contable de shapes

Una microred recibe dos características, tiene dos unidades ocultas y produce una probabilidad. Antes del código, complete mentalmente cada shape.

In [ ]:
ledger = pd.DataFrame([
    {'objeto': 'X', 'shape': '(B, 2)', 'significado': 'lote de entradas'},
    {'objeto': 'W1', 'shape': '(2, 2)', 'significado': 'pesos de la capa oculta'},
    {'objeto': 'b1', 'shape': '(2,)', 'significado': 'sesgos ocultos'},
    {'objeto': 'A1', 'shape': '(B, 2)', 'significado': 'representación oculta'},
    {'objeto': 'W2', 'shape': '(2, 1)', 'significado': 'pesos de salida'},
    {'objeto': 'b2', 'shape': '(1,)', 'significado': 'sesgo de salida'},
    {'objeto': 'y_hat', 'shape': '(B, 1)', 'significado': 'probabilidad estimada'},
])
ledger

## 3. Abramos la caja: forward pass con NumPy

Asignamos valores conocidos. No estamos entrenando: estamos calculando exactamente la función definida por la arquitectura.

In [ ]:
X_micro = np.array([[1.0, 2.0], [-1.0, 0.5]], dtype=np.float32)
W1 = np.array([[0.5, -0.4], [1.0, 0.8]], dtype=np.float32)
b1 = np.array([0.1, -0.2], dtype=np.float32)
W2 = np.array([[1.2], [-0.7]], dtype=np.float32)
b2 = np.array([0.05], dtype=np.float32)

z1 = X_micro @ W1 + b1
a1 = np.maximum(z1, 0.0)
z2 = a1 @ W2 + b2
y_numpy = 1.0 / (1.0 + np.exp(-z2))

for name, value in {'X': X_micro, 'Z1': z1, 'A1': a1, 'Z2': z2, 'y_hat': y_numpy}.items():
    print(f'{name:5s} shape={value.shape}\n{value}\n')

**Comprobación.** Identifique qué valores anuló ReLU. ¿Cómo afecta eso a la contribución de cada unidad oculta?

## 4. La misma función en Keras

Construimos la arquitectura y cargamos exactamente los mismos pesos. Si comprendimos la operación, ambas salidas deben coincidir.

In [ ]:
micro_model = keras.Sequential([
    keras.Input(shape=(2,), name='entrada'),
    keras.layers.Dense(2, activation='relu', name='oculta'),
    keras.layers.Dense(1, activation='sigmoid', name='salida'),
])
micro_model.get_layer('oculta').set_weights([W1, b1])
micro_model.get_layer('salida').set_weights([W2, b2])
y_keras = micro_model(X_micro, training=False).numpy()

print('Parámetros esperados:', 2 * 2 + 2 + 2 * 1 + 1)
print('Parámetros de Keras:', micro_model.count_params())
print('Salidas iguales:', np.allclose(y_numpy, y_keras, atol=1e-6))
micro_model.summary()

## 5. Problema no lineal y particiones

Generamos dos medias lunas. Separamos test una sola vez y obtenemos validación únicamente desde el conjunto de desarrollo.

In [ ]:
X, y = make_moons(n_samples=900, noise=0.22, random_state=RANDOM_STATE)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=RANDOM_STATE
)
scaler = StandardScaler().fit(X_train)
X_train_s, X_val_s, X_test_s = map(scaler.transform, (X_train, X_val, X_test))

plt.figure(figsize=(6, 4))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='coolwarm', s=16, alpha=.75)
plt.title('Datos de entrenamiento')
plt.xlabel('x1'); plt.ylabel('x2'); plt.show()

## 6. Baseline lineal

La regresión logística establece cuánto logra una frontera lineal. Test permanece sin tocar hasta la comparación final.

In [ ]:
baseline = LogisticRegression(random_state=RANDOM_STATE).fit(X_train_s, y_train)
baseline_val_prob = baseline.predict_proba(X_val_s)[:, 1]
print({'val_loss': log_loss(y_val, baseline_val_prob), 'val_accuracy': accuracy_score(y_val, baseline_val_prob >= .5)})

## 7. Primer MLP completo

Usamos una arquitectura pequeña y SGD. En sesiones posteriores compararemos activaciones, optimizadores y regularización de forma controlada.

In [ ]:
keras.utils.set_random_seed(RANDOM_STATE)
model = keras.Sequential([
    keras.Input(shape=(2,), name='entrada'),
    keras.layers.Dense(8, activation='relu', name='representacion'),
    keras.layers.Dense(1, activation='sigmoid', name='probabilidad'),
])
model.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.05),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)
history = model.fit(
    X_train_s, y_train, validation_data=(X_val_s, y_val),
    epochs=80, batch_size=32, verbose=0, shuffle=True,
)
pd.DataFrame(history.history).tail()

In [ ]:
curves = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
curves[['loss', 'val_loss']].plot(ax=axes[0], title='Pérdida por época')
curves[['accuracy', 'val_accuracy']].plot(ax=axes[1], title='Accuracy por época')
for ax in axes: ax.set_xlabel('época'); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()

## 8. Evaluación final y lectura

Solo después de fijar la configuración observamos test. Reportamos baseline y MLP bajo la misma partición.

In [ ]:
baseline_test_prob = baseline.predict_proba(X_test_s)[:, 1]
mlp_test_prob = model.predict(X_test_s, verbose=0).ravel()
comparison = pd.DataFrame({
    'modelo': ['Regresión logística', 'MLP 2→8→1'],
    'test_log_loss': [log_loss(y_test, baseline_test_prob), log_loss(y_test, mlp_test_prob)],
    'test_accuracy': [accuracy_score(y_test, baseline_test_prob >= .5), accuracy_score(y_test, mlp_test_prob >= .5)],
})
comparison

## 9. Cierre y reto

1. Construya el libro contable para `Dense(8) → Dense(1)`.
2. Explique por qué el MLP puede representar una frontera que la regresión logística no.
3. Identifique en las curvas evidencia de aprendizaje, estabilidad y posible separación train–validación.
4. Cambie únicamente el número de unidades ocultas a 2. Prediga el efecto antes de ejecutar y compare sin tocar test.

**Mensaje de salida:** Keras automatiza el cálculo y la diferenciación; comprender shapes, parámetros y evidencia evita tratar el modelo como una caja negra.